# `move_to_delta` Hardware Validation (BACKLOG 6A.1)

Validate that `move_to_delta(dx, dy, dz)` traces a **straight line** for diagonal moves,
instead of the staircase produced by chained axis-aligned primitives.

Frame: `+X = forward`, `+Y = left`, `+Z = up` (meters).

## 1. Connect to follower arm (raw lerobot SDK)

Same pattern as `basic_debug_test.ipynb`.

In [1]:
from lerobot.robots.so_follower import SOFollowerRobotConfig
from lerobot.robots import make_robot_from_config

config = SOFollowerRobotConfig(
    id="decras_follower",
    port="/dev/ttyACM0",
    disable_torque_on_disconnect=True,
    max_relative_target=None,
    use_degrees=True,
)

robot = make_robot_from_config(config)
robot.connect(calibrate=True)
robot.get_observation()

/home/alexandre/IA/DecRAS/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'shoulder_pan.pos': -18.54945054945055,
 'shoulder_lift.pos': -62.94505494505494,
 'elbow_flex.pos': 41.84615384615385,
 'wrist_flex.pos': 71.95604395604396,
 'wrist_roll.pos': 0.21978021978021978,
 'gripper.pos': 0.7575757575757576}

## 2. Wrap with `LeRobotInterface`

We use `__new__` to bypass `__init__` (which would open a second serial connection)
and attach the already-connected `robot`. This gives us `move_cartesian_delta` —
the exact code path the MCP `move_to_delta` tool calls.

In [2]:
import os
os.environ["SIMULATE"] = "false"   # MUST be set before reloading the modules

import importlib
import mcp_server.config as _cfg
import mcp_server.robot.kinematics as _kin
import mcp_server.robot.lerobot as _lr
importlib.reload(_cfg); importlib.reload(_kin); importlib.reload(_lr)
from mcp_server.robot.lerobot import LeRobotInterface

assert _lr.SIMULATE is False, f"SIMULATE still {_lr.SIMULATE} — restart kernel"
print("SIMULATE =", _lr.SIMULATE)

iface = LeRobotInterface.__new__(LeRobotInterface)
iface._port = "/dev/ttyACM0"
iface._robot_id = "decras_follower"
iface._robot = robot          # reuse the connected lerobot robot
iface._is_connected = True
iface._position = [0.2, 0.0, 0.15]
iface._gripper_open = True
iface._holding = None
iface._last_action = "init"
iface._last_status = "complete"

print("EE now: ", iface.get_ee_position())
print("Joints: ", iface.get_joint_positions())

SIMULATE = False
EE now:  [0.19048934868247602, 0.05084567024738207, 0.1137963937536101]
Joints:  {'shoulder_pan': -18.54945054945055, 'shoulder_lift': -62.857142857142854, 'elbow_flex': 42.81318681318681, 'wrist_flex': 71.95604395604396, 'wrist_roll': 0.21978021978021978, 'gripper': 0.7575757575757576}


  -base_link_2 collides with shoulder_link_2
  -shoulder_link_0 collides with upper_arm_link_1
  -upper_arm_link_0 collides with lower_arm_link_0
  -lower_arm_link_2 collides with wrist_link_1
  -wrist_link_0 collides with gripper_link_1
  -gripper_link_0 collides with moving_jaw_so101_v1_link_0


## 3. Go to WORK position

Safe gentle pose with the EE in free space. All deltas measured from here.

In [3]:
import time

WORK = {
    "shoulder_pan.pos":  -18.0,
    "shoulder_lift.pos": -60.0,
    "elbow_flex.pos":     37.0,
    "wrist_flex.pos":     74.0,
    "wrist_roll.pos":      0.0,
    "gripper.pos":         0.0,
}
robot.send_action(WORK)
time.sleep(1.5)

ee_work = iface.get_ee_position()
print(f"EE at WORK: [{ee_work[0]:.4f}, {ee_work[1]:.4f}, {ee_work[2]:.4f}] m")

EE at WORK: [0.1885, 0.0502, 0.1111] m


## 4. Sanity check — single-axis deltas

Each call moves ~5cm along ONE axis. Confirms axes/signs before testing diagonals.
- forward/back = away/toward base (X)
- left/right = your left/right looking at the arm from behind (Y)
- up/down = vertical (Z)

## 4b. Diagnostic — `move_to_delta` reproduced manually using the working approach

Same goal as `iface.move_cartesian_delta(+0.05, 0, -0.03)`, but built like the working
square in `basic_debug_test.ipynb`:

1. **Settle WORK pose** — send the same setpoint a few times so the arm fully reaches it
   (counters servo droop).
2. **Seed IK from the WORK FK** (not from `get_observation()`), so we never read a
   mid-motion / sagged position.
3. **Discretize the path** in pure CPU (5 waypoints along the diagonal).
4. **Send each waypoint** via raw `robot.send_action()` with a real sleep between (0.3s).
5. **Active-hold the final pose** for 1s before measuring.

If THIS cell traces correctly, the bug is in `move_cartesian_delta` (server-side fix).
If this also fails, the bug is deeper (FK/IK, calibration, or hardware).

In [ ]:
import numpy as np
from mcp_server.robot.kinematics import joints_to_cartesian, cartesian_to_joints

WORK_JOINTS = {k.replace(".pos", ""): v for k, v in WORK.items()}
WORK_ARM = {k: v for k, v in WORK_JOINTS.items() if k != "gripper"}

# 1. Settle into WORK by repeating the setpoint (active hold against gravity)
print("Settling into WORK ...")
for _ in range(8):
    robot.send_action(WORK)
    time.sleep(0.15)

# 2. Seed IK from the WORK FK (NOT from the actual sagged hardware reading)
ee_seed = np.array(joints_to_cartesian(WORK_ARM))
print(f"WORK FK seed: [{ee_seed[0]:.4f}, {ee_seed[1]:.4f}, {ee_seed[2]:.4f}] m")
print(f"HW reads:     {iface.get_ee_position()}  (sag → ignore for IK seed)")

# 3. Plan 5 waypoints along the diagonal: +5cm X, 0cm Y, -3cm Z
DELTA = np.array([+0.05, 0.0, -0.03])
N = 5
waypoints = [ee_seed + DELTA * (i / N) for i in range(1, N + 1)]

# Solve IK for each waypoint, seeding from previous IK solution (NOT from hw)
seed = dict(WORK_ARM)
ik_plan = []
for i, wp in enumerate(waypoints):
    ik = cartesian_to_joints(wp[0], wp[1], wp[2], seed_hw_joints=seed)
    fk_check = joints_to_cartesian(ik)
    err_mm = np.linalg.norm(np.array(fk_check) - wp) * 1000
    print(f"  wp{i+1} target={[round(v,4) for v in wp.tolist()]}  "
          f"IK→FK err={err_mm:.2f}mm")
    ik_plan.append(ik)
    seed = ik

# 4. Execute: send each waypoint, then active-hold
input("\nPress ENTER to execute the planned trajectory...")

for i, ik in enumerate(ik_plan):
    action = {f"{k}.pos": v for k, v in ik.items()}
    action["gripper.pos"] = 0.0
    robot.send_action(action)
    time.sleep(0.3)

# 5. Active-hold the final pose so we measure where it ACTUALLY ends up
final_action = {f"{k}.pos": v for k, v in ik_plan[-1].items()}
final_action["gripper.pos"] = 0.0
for _ in range(8):
    robot.send_action(final_action)
    time.sleep(0.15)

p_end = np.array(iface.get_ee_position())
delta_meas = p_end - ee_seed
print(f"\nStart (FK seed): {ee_seed}")
print(f"End   (HW FK):   {p_end}")
print(f"Delta measured:  [{delta_meas[0]:+.4f}, {delta_meas[1]:+.4f}, {delta_meas[2]:+.4f}]")
print(f"Delta expected:  [+0.0500, +0.0000, -0.0300]")
print(f"Error (mm):      [{(delta_meas[0]-DELTA[0])*1000:+.1f}, "
      f"{(delta_meas[1]-DELTA[1])*1000:+.1f}, "
      f"{(delta_meas[2]-DELTA[2])*1000:+.1f}]")

In [4]:
def go(dx, dy, dz, label):
    p0 = iface.get_ee_position()
    res = iface.move_cartesian_delta(dx, dy, dz)
    p1 = iface.get_ee_position()
    measured = [p1[i] - p0[i] for i in range(3)]
    print(f"{label:18s}  cmd=[{dx:+.3f},{dy:+.3f},{dz:+.3f}]  "
          f"meas=[{measured[0]:+.3f},{measured[1]:+.3f},{measured[2]:+.3f}]  "
          f"status={res.get('status')}")
    time.sleep(1.0)

D = 0.02
go(+D, 0, 0, "forward (+X)")
go(-D, 0, 0, "back    (-X)")
go(0, +D, 0, "left    (+Y)")
go(0, -D, 0, "right   (-Y)")
go(0, 0, +D, "up      (+Z)")
go(0, 0, -D, "down    (-Z)")

forward (+X)        cmd=[+0.020,+0.000,+0.000]  meas=[+0.008,-0.001,-0.006]  status=complete
back    (-X)        cmd=[-0.020,+0.000,+0.000]  meas=[-0.014,-0.001,-0.006]  status=complete
left    (+Y)        cmd=[+0.000,+0.020,+0.000]  meas=[-0.003,+0.010,-0.005]  status=complete
right   (-Y)        cmd=[+0.000,-0.020,+0.000]  meas=[-0.002,-0.012,-0.005]  status=complete
up      (+Z)        cmd=[+0.000,+0.000,+0.020]  meas=[-0.004,-0.001,+0.002]  status=complete
down    (-Z)        cmd=[+0.000,+0.000,-0.020]  meas=[-0.005,-0.002,-0.019]  status=complete


## 5. The actual test — diagonal XZ

**5cm forward + 3cm down in a single call.** Should trace a straight diagonal,
not a staircase. Watch the gripper tip carefully.

In [6]:
robot.send_action(WORK)
time.sleep(1.5)
p0 = iface.get_ee_position()
print(f"Start EE: [{p0[0]:.4f}, {p0[1]:.4f}, {p0[2]:.4f}]")

input("Press ENTER to fire the diagonal move...")

res = iface.move_cartesian_delta(+0.05, 0.0, -0.03)
p1 = iface.get_ee_position()
print(f"End   EE: [{p1[0]:.4f}, {p1[1]:.4f}, {p1[2]:.4f}]")
print(f"Delta:    [{p1[0]-p0[0]:+.4f}, {p1[1]-p0[1]:+.4f}, {p1[2]-p0[2]:+.4f}]")
print(f"Expected: [+0.0500, +0.0000, -0.0300]")
print(f"Status:   {res.get('status')}")

Start EE: [0.1886, 0.0487, 0.1089]
End   EE: [0.2061, 0.0474, 0.0756]
Delta:    [+0.0175, -0.0013, -0.0333]
Expected: [+0.0500, +0.0000, -0.0300]
Status:   complete


## 6. Diagonal YZ

**4cm left + 4cm up** — diagonal in YZ plane.

In [7]:
robot.send_action(WORK)
time.sleep(1.5)
p0 = iface.get_ee_position()

input("Press ENTER for YZ diagonal (+4cm left, +4cm up)...")

res = iface.move_cartesian_delta(0.0, +0.04, +0.04)
p1 = iface.get_ee_position()
print(f"Delta:    [{p1[0]-p0[0]:+.4f}, {p1[1]-p0[1]:+.4f}, {p1[2]-p0[2]:+.4f}]")
print(f"Expected: [+0.0000, +0.0400, +0.0400]")
print(f"Status:   {res.get('status')}")

Delta:    [-0.0096, +0.0199, -0.0004]
Expected: [+0.0000, +0.0400, +0.0400]
Status:   complete


## 7. Full 3D diagonal

**3cm forward + 3cm left + 3cm up** — all three axes at once.

In [8]:
robot.send_action(WORK)
time.sleep(1.5)
p0 = iface.get_ee_position()

input("Press ENTER for full 3D diagonal...")

res = iface.move_cartesian_delta(+0.03, +0.03, +0.03)
p1 = iface.get_ee_position()
print(f"Delta:    [{p1[0]-p0[0]:+.4f}, {p1[1]-p0[1]:+.4f}, {p1[2]-p0[2]:+.4f}]")
print(f"Expected: [+0.0300, +0.0300, +0.0300]")
print(f"Status:   {res.get('status')}")

Delta:    [+0.0096, +0.0137, -0.0005]
Expected: [+0.0300, +0.0300, +0.0300]
Status:   complete


## 8. Disconnect

In [9]:
robot.send_action(WORK)
time.sleep(1.0)
robot.disconnect()
print("Disconnected.")

Disconnected.
